# VadCLIP · Cắt Đặc Trưng & Nhất Quán Theo Dịch Chuyển — Bản Kaggle

Notebook này chạy **hai khối thí nghiệm tách rời nhau**, cộng một khối đối chứng.

| Khối | Cắt đặc trưng? | `L_shift`? | Câu hỏi |
|---|---|---|---|
| **Ctrl** | không | không | Mốc so. Đây đúng là VadCLIP gốc. |
| **A** | có, nhiều kiểu | **không** | Chỉ riêng việc coi khung nhìn dịch là *dữ liệu tăng cường* có giúp không? Kiểu cắt nào tốt nhất? |
| **B** | một kiểu cố định | **có** | Trọng số λ nên tính thế nào cho không phải đoán mò? |

Tách hai khối là điểm khác quan trọng nhất so với bản cũ. Trước đây mọi lần chạy đều bật
khung nhìn dịch, nhưng ba hàm mất mát gốc chỉ tính trên khung nhìn đầy đủ — nên `λ = 0`
**không phải** là "có augment mà không có loss", nó đơn giản là VadCLIP gốc với một
tensor bị nhân với 0. Khối A ở đây dùng cờ mới `--augment-task-loss`, khiến `L₁`/`L₂`
được tính trên **cả hai** khung nhìn. Đó mới là augmentation thật.

---

## Quy ước tên: ở đây có **ba** thứ khác nhau cùng tên "AUC"

Đây là chỗ bản cũ gây rối nhất, nên đọc kỹ một lần rồi khỏi phải đoán nữa.

| Tên trong notebook này | Là gì | Đo trên |
|---|---|---|
| `auc_branch_c` | ROC-AUC của nhánh phân lớp nhị phân. Log gốc in là `AUC1`. **Đây là chỉ số chính**, cũng là thứ dùng để chọn checkpoint. | 290 video test, toàn bộ khung |
| `auc_branch_a` | ROC-AUC của nhánh căn chỉnh thị giác–văn bản. Log gốc in là `AUC2`. | 290 video test, toàn bộ khung |
| `auc_spread_shift` | **Không cùng loại với hai cái trên.** Là biên độ dao động AUC khi dịch đầu vào đi Δ bước, do `ucf_shift_sensitivity.py` tính. | 266 video, **chỉ vùng chồng lấn** |

Ba con số này nằm ở **ba bảng khác nhau** trong mục 11. Đừng đặt chúng cạnh nhau trong
cùng một cột — `auc_spread_shift` chạy trên tập con và vùng con, nên so trực tiếp với
`auc_branch_c` là so hai thứ khác nhau.

Và với mỗi lần chạy có **hai** thời điểm khác nhau:

- `sel_*` — số của **lần chấm có AUC cao nhất**, tức đúng bộ trọng số được ghi ra file.
- `end_*` — số ở **cuối epoch cuối**, chỉ để tham khảo.

Notebook này đặt `--select-metric auc`, nên **file trọng số bạn nhận được là `sel_*`**.

> ⚠️ **Nói rõ một lần về luật chọn này.** `--select-metric auc` chọn checkpoint theo AUC
> trên chính tập test, qua khoảng 120 lần chấm. Đó là chọn-mô-hình-trên-test, và vòng 2
> đã đo được rằng luật này thổi phồng chênh lệch giữa các lần chạy lên khoảng một bậc.
> Bạn đã yêu cầu lưu theo AUC cao nhất nên notebook làm đúng vậy, nhưng mục 11 in **cả**
> `sel_*` lẫn `end_*` để bạn nhìn được phần nào là tín hiệu, phần nào là do luật chọn.

## 1. Cấu Hình & Danh Mục Lần Chạy

Tự dò dataset trong `/kaggle/input`. Dò sai thì điền tay vào ba biến `*_OVERRIDE`.

Code phải nằm ở nơi ghi được (`ucf_train_augment.py` ghi `model/` theo thư mục hiện
hành), mà `/kaggle/input` là chỉ đọc — nên cell này copy `src/` và `list/` sang
`/kaggle/working/vadclip`.

**Phần cần bạn sửa nằm ở cuối cell**: biến `RUNS_THIS_SESSION`. Một phiên Kaggle chỉ có
12 giờ và toàn bộ danh mục cần nhiều hơn thế, nên bạn chạy vài lần chạy mỗi phiên rồi
Save Version, phiên sau gắn output cũ làm input và notebook sẽ tự bỏ qua những gì đã xong.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK       = Path('/kaggle/working')
TEMP       = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')

# --- Code lấy từ đâu ------------------------------------------------------------------
# 'auto'    : có dataset code trong /kaggle/input thì dùng, không có thì clone GitHub
# 'github'  : luôn clone, kể cả khi có dataset
# 'dataset' : luôn dùng dataset, không clone
CODE_SOURCE   = 'auto'
# Chi de IN RA trong thong bao goi y. KHONG dieu khien viec do duong dan --
# cai do la FEATURE_OVERRIDE ben duoi.
FEATURE_DATASET_HINT = 'beosngu/ucf-crime-vadclip-features'   # 14 GB, cong khai
GITHUB_REPO   = 'https://github.com/vngclinh/Finetune-VadCLIP.git'
GITHUB_BRANCH = 'main'

# Điền tay nếu tự dò sai. Ví dụ: Path('/kaggle/input/vadclip-shift')
CODE_OVERRIDE    = None
FEATURE_OVERRIDE = None
CKPT_OVERRIDE    = None


def walk_dirs(root, maxdepth=8):
    """Duyet thu muc theo be rong, CO di xuyen symlink.

    Khong dung rglob: `**` cua pathlib goi is_dir(follow_symlinks=False), tuc la no
    co y bo qua thu muc symlink. Kaggle mount dataset bang symlink, nen rglob khong
    bao gio nhin thay gi ben trong /kaggle/input. iterdir() + is_dir() thi di xuyen
    binh thuong.
    """
    if not root.exists():
        return
    seen = set()
    queue = [(root, 0)]
    while queue:
        directory, depth = queue.pop(0)
        try:
            key = directory.resolve()
        except OSError:
            key = directory
        if key in seen:
            continue
        seen.add(key)
        yield directory
        if depth >= maxdepth:
            continue
        try:
            queue.extend((child, depth + 1)
                         for child in sorted(directory.iterdir()) if child.is_dir())
        except (PermissionError, OSError):
            pass


def find_in_input(*markers, maxdepth=8):
    """Thu muc con cua /kaggle/input chua du cac duong dan danh dau."""
    for directory in walk_dirs(INPUT_ROOT, maxdepth):
        if all((directory / m).exists() for m in markers):
            return directory
    return None


def clone_repo():
    """Clone repo vào /kaggle/working và trả về thư mục chứa src/ và list/."""
    clone_dir = WORK / 'repo'
    if (clone_dir / '.git').exists():
        print('Đã có repo, cập nhật về bản mới nhất ...')
        subprocess.run(['git', '-C', str(clone_dir), 'fetch', '--depth', '1',
                        'origin', GITHUB_BRANCH], check=True)
        subprocess.run(['git', '-C', str(clone_dir), 'reset', '--hard',
                        f'origin/{GITHUB_BRANCH}'], check=True)
    else:
        print('Clone', GITHUB_REPO, '...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH,
                        GITHUB_REPO, str(clone_dir)], check=True)
    sha = subprocess.run(['git', '-C', str(clone_dir), 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print('Commit:', sha)
    return clone_dir / 'VadCLIP'


CODE_FROM_GITHUB = False
CODE_ROOT = CODE_OVERRIDE
if CODE_ROOT is None and CODE_SOURCE != 'github':
    CODE_ROOT = find_in_input('src/ucf_train_augment.py',
                              'list/ucf_CLIP_rgbtest_relative.csv')
    if CODE_ROOT is None:
        CODE_ROOT = find_in_input('src/ucf_train_augment.py')  # thiếu list -> preflight báo
if CODE_ROOT is None and CODE_SOURCE in ('auto', 'github'):
    CODE_ROOT = clone_repo()
    CODE_FROM_GITHUB = True


def find_feature_root():
    """Thu muc chua cac thu muc lop."""
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir() and (directory / 'Vandalism').is_dir():
            return directory, 'thay Abuse + Vandalism'
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir():
            return directory, 'chi thay Abuse'
    for directory in walk_dirs(INPUT_ROOT):
        try:
            children = [d for d in directory.iterdir() if d.is_dir()]
        except (PermissionError, OSError):
            continue
        with_npy = [d for d in children if next(d.glob('*.npy'), None) is not None]
        if len(with_npy) >= 10:
            return directory, f'{len(with_npy)} thu muc con co file .npy'
    return None, None


def print_input_tree(levels=6):
    """In cay /kaggle/input de nhin ra ngay cau truc that cua dataset."""
    print()
    print('=' * 78)
    print('/kaggle/input dang co gi:')
    if not INPUT_ROOT.exists():
        print('   (khong ton tai -- chua Add Input dataset nao)')
        return
    entries = sorted(INPUT_ROOT.iterdir())
    if not entries:
        print('   (rong -- chua Add Input dataset nao)')
        return

    def walk(path, depth):
        if depth > levels:
            return
        try:
            children = sorted(path.iterdir())
        except (PermissionError, OSError):
            return
        for child in children[:15]:
            npy = len(list(child.glob('*.npy'))) if child.is_dir() else 0
            mark = f'  <- {npy} file .npy' if npy else ''
            print('   ' + '   ' * depth + child.name + ('/' if child.is_dir() else '') + mark)
            if child.is_dir() and not npy:
                walk(child, depth + 1)
        if len(children) > 15:
            print('   ' + '   ' * depth + f'... con {len(children) - 15} muc nua')
    walk(INPUT_ROOT, 0)
    print('=' * 78)


if FEATURE_OVERRIDE is not None:
    FEATURE_ROOT, how = FEATURE_OVERRIDE, 'FEATURE_OVERRIDE dat tay'
else:
    FEATURE_ROOT, how = find_feature_root()

if FEATURE_ROOT is None:
    print('CHUA DO RA DATASET FEATURE.')
    print('Panel ben phai -> Add Input -> Datasets -> dan:', FEATURE_DATASET_HINT)
    print('Da add roi ma van bao the nay thi thu Run -> Restart & Clear Cell Outputs.')
    print_input_tree()
    print("    FEATURE_OVERRIDE = Path('/kaggle/input/<ten-dataset>')")
else:
    print('Feature do ra bang:', how)

CKPT_ROOT = CKPT_OVERRIDE or find_in_input('model_ucf.pth') or CODE_ROOT

if CODE_ROOT is None:
    print_input_tree()
    raise FileNotFoundError(
        'Không tìm thấy code. Đặt CODE_SOURCE = "github" để clone, hoặc điền CODE_OVERRIDE.')

# --- Code sang nơi ghi được -----------------------------------------------------------
PROJECT  = WORK / 'vadclip'
SRC_DIR  = PROJECT / 'src'
LIST_DIR = PROJECT / 'list'
if not SRC_DIR.exists():
    print('Copy code sang thư mục ghi được ...')
    shutil.copytree(CODE_ROOT / 'src', SRC_DIR)
if not LIST_DIR.exists() and (CODE_ROOT / 'list').exists():
    shutil.copytree(CODE_ROOT / 'list', LIST_DIR)
# __pycache__ đi theo từ dataset sẽ che mất file .py mới. Xoá cho chắc.
for cache in SRC_DIR.rglob('__pycache__'):
    shutil.rmtree(cache, ignore_errors=True)

PAPER_MODEL = (CKPT_ROOT / 'model_ucf.pth') if CKPT_ROOT else None

# --- Nơi ghi kết quả ------------------------------------------------------------------
# Giữ lại (vào Output): trọng số đã chọn, log, CSV, bảng độ nhạy.
# Vứt đi (sang /kaggle/temp): checkpoint theo epoch, model_cur, checkpoint trung gian.
RESULT_DIR = WORK / 'results'
LOG_DIR    = RESULT_DIR / 'logs'
MODEL_DIR  = WORK / 'models'
SCRATCH    = TEMP / 'vadclip_scratch'
for directory in (RESULT_DIR, LOG_DIR, MODEL_DIR, SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)

METRICS_CSV = str(RESULT_DIR / 'shift_kaggle_metrics.csv')

TRAIN_LIST = str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')
TEST_LIST  = str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv')
GT_ARGS = [
    '--gt-path',         str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path',   str(LIST_DIR / 'gt_label_ucf.npy'),
]

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)
PY = [sys.executable, '-u']

In [ ]:
# ======================================================================================
#  CẤU HÌNH CHUNG — giống nhau ở MỌI lần chạy, để thứ duy nhất khác nhau là cái đang thử
# ======================================================================================
SEED           = 234
MAX_EPOCH      = 10
LR             = '2e-5'
BATCH_SIZE     = 64      # mặc định script; ở đây dùng cho ước lượng thời gian
NUM_WORKERS    = 4
EVAL_STEPS     = 1280    # cadence của ucf_train.py gốc, ~12 lần chấm mỗi epoch
USE_PRETRAINED = False   # train từ đầu
WARMUP_EPOCHS  = 1       # ramp tuyến tính của lambda
BRANCH         = 'c'     # nhánh bị ràng buộc bởi L_shift

# Bạn yêu cầu: lưu checkpoint theo lần chấm có AUC CAO NHẤT, không phải trọng số cuối.
SELECT_METRIC  = 'auc'

# Khối B dùng chung một kiểu cắt, để thứ duy nhất thay đổi là cách tính lambda.
BASE_CUT = 'head26'

# Ngân sách một phiên Kaggle. Phiên cứng 12 giờ, chừa chỗ cho mục 11-13.
TIME_BUDGET_HOURS = 10.0

# ======================================================================================
#  SÁU KIỂU CẮT
# ======================================================================================
#  offset  : số ô lưới cố định
#  ratio   : cắt theo % độ dài hợp lệ CỦA CHÍNH VIDEO đó (ratio > 0 thì offset bị bỏ qua)
#  direction: head = bỏ phần đầu | tail = đệm vào đầu, đẩy nội dung lùi | both = bốc mỗi item
#  random  : bốc độ lớn trong [1, max] thay vì luôn dùng max
CUTS = {
    'nocut':    dict(nhan='không cắt',              skip=True),
    'head26':   dict(nhan='bỏ 26 ô đầu',            offset=26,   direction='head'),
    'tail26':   dict(nhan='đệm 26 ô vào đầu',       offset=26,   direction='tail'),
    'both26r':  dict(nhan='±(1..26) ngẫu nhiên',    offset=26,   direction='both', random=True),
    'ratio10':  dict(nhan='±10% độ dài video',      ratio=0.10,  direction='both'),
    'ratio20r': dict(nhan='±(1..20%) ngẫu nhiên',   ratio=0.20,  direction='both', random=True),
}

# ======================================================================================
#  SÁU CÁCH TÍNH LAMBDA
# ======================================================================================
#  off    : không có L_shift
#  fixed  : đặt tay một số tuyệt đối
#  auto   : giải lambda sao cho L_shift chiếm tỉ lệ `ratio` của hàm mục tiêu,
#           đo theo `basis` ('loss' = giá trị loss, 'grad' = chuẩn gradient)
LAMBDAS = {
    'off':            dict(nhan='không có L_shift',                 kind='off'),
    'fixed001':       dict(nhan='tay, tuyệt đối 0,01',              kind='fixed', value=0.01),
    'loss001':        dict(nhan='tỉ lệ loss 1%, giải 1 lần',        kind='auto', ratio=0.01, basis='loss'),
    'loss001_recal':  dict(nhan='tỉ lệ loss 1%, giải lại mỗi epoch', kind='auto', ratio=0.01, basis='loss', recal=True),
    'grad001':        dict(nhan='tỉ lệ gradient 1%',                kind='auto', ratio=0.01, basis='grad'),
    'grad003':        dict(nhan='tỉ lệ gradient 3%',                kind='auto', ratio=0.03, basis='grad'),
    'grad010':        dict(nhan='tỉ lệ gradient 10%',               kind='auto', ratio=0.10, basis='grad'),
}

# ======================================================================================
#  DANH MỤC LẦN CHẠY — mỗi dòng là một lần chạy, không có gì ẩn
# ======================================================================================
RUN_REGISTRY = {}


def dang_ky(tag, khoi, cut, augment, lam):
    RUN_REGISTRY[tag] = dict(tag=tag, khoi=khoi, cut=cut, augment=augment, lam=lam)


# Đối chứng: không cắt, không augment, không L_shift. Đây đúng là VadCLIP gốc.
dang_ky('ctrl_nocut', 'Ctrl', 'nocut', augment=False, lam='off')

# Khối A: cắt như dữ liệu tăng cường, KHÔNG có L_shift.
for _cut in ('head26', 'tail26', 'both26r', 'ratio10', 'ratio20r'):
    dang_ky(f'aug_{_cut}', 'A', _cut, augment=True, lam='off')

# Khối B: một kiểu cắt cố định, KHÔNG augment, chỉ đổi cách tính lambda.
for _lam in ('fixed001', 'loss001', 'loss001_recal', 'grad001', 'grad003', 'grad010'):
    dang_ky(f'lam_{_lam}', 'B', BASE_CUT, augment=False, lam=_lam)

# ======================================================================================
#  ⇩⇩⇩  SỬA Ở ĐÂY: phiên này chạy những lần chạy nào  ⇩⇩⇩
# ======================================================================================
#  Toàn bộ danh mục là 12 lần chạy × ~2-3 giờ, không vừa một phiên 12 giờ.
#  Gợi ý chia phiên (mục 7 sẽ kiểm lại bằng số đo thật):
#
#    Phiên 1: ctrl_nocut, aug_head26, aug_tail26, aug_both26r
#    Phiên 2: aug_ratio10, aug_ratio20r, lam_fixed001, lam_loss001
#    Phiên 3: lam_loss001_recal, lam_grad001, lam_grad003, lam_grad010
#
#  Sau mỗi phiên: Save Version. Phiên sau Add Input chính notebook này, mục 1 sẽ tìm
#  thấy model cũ và mục 9/10 sẽ bỏ qua chúng.
RUNS_THIS_SESSION = [
    'ctrl_nocut',
    'aug_head26',
    'aug_tail26',
    'aug_both26r',
]

unknown = [t for t in RUNS_THIS_SESSION if t not in RUN_REGISTRY]
if unknown:
    raise KeyError(f'Tag không có trong RUN_REGISTRY: {unknown}')

# --- Kết quả của phiên trước, nếu output cũ được gắn làm input -------------------------
PRIOR_METRICS = [str(f) for d in walk_dirs(INPUT_ROOT)
                 for f in sorted(d.glob('shift_kaggle_metrics.csv'))]
PRIOR_MODEL_DIRS = [d for d in walk_dirs(INPUT_ROOT)
                    if next(d.glob('model_*.pth'), None) is not None]
PRIOR_RESULT_DIRS = [d for d in walk_dirs(INPUT_ROOT)
                     if d.name.startswith('shift_sens_')]


def duong_dan_model(tag):
    """Trọng số của một lần chạy: phiên này trước, rồi mới tới phiên trước."""
    here = MODEL_DIR / f'model_{tag}.pth'
    if here.exists():
        return here
    for directory in PRIOR_MODEL_DIRS:
        candidate = directory / f'model_{tag}.pth'
        if candidate.exists():
            return candidate
    return None


print('Nguồn code :', 'GitHub (' + GITHUB_BRANCH + ')' if CODE_FROM_GITHUB else 'Dataset ' + str(CODE_ROOT))
print('Code       :', SRC_DIR)
print('Feature    :', FEATURE_ROOT)
print('Checkpoint :', PAPER_MODEL)
print('Kết quả    :', RESULT_DIR)
print()
print(f'Cấu hình chung: seed {SEED} | {MAX_EPOCH} epoch | lr {LR} | lô {BATCH_SIZE} '
      f'| chấm mỗi {EVAL_STEPS} mẫu | select-metric {SELECT_METRIC}')
print()
print('=' * 94)
print(f'{"tag":<20} {"khối":<5} {"cắt":<26} {"augment":<8} {"lambda"}')
print('=' * 94)
for tag, run in RUN_REGISTRY.items():
    da_co = duong_dan_model(tag) is not None
    chon = tag in RUNS_THIS_SESSION
    dau = 'đã xong' if da_co else ('PHIÊN NÀY' if chon else '')
    print(f'{tag:<20} {run["khoi"]:<5} {CUTS[run["cut"]]["nhan"]:<26} '
          f'{"có" if run["augment"] else "không":<8} {LAMBDAS[run["lam"]]["nhan"]:<32} {dau}')
print('=' * 94)
print()
print('Phiên này sẽ chạy:', [t for t in RUNS_THIS_SESSION if duong_dan_model(t) is None])
if PRIOR_METRICS:
    print('Tìm thấy CSV của phiên trước:', PRIOR_METRICS)

## 2. Dependencies

Ảnh Kaggle đã có torch, sklearn, scipy, pandas, matplotlib. Chỉ thiếu `ftfy` và `regex`
mà tokenizer của CLIP cần. Cần **Internet: On**.

In [ ]:
!pip -q install ftfy regex

## 3. Nạp Sẵn Trọng Số CLIP

`model.py` gọi `clip.load("ViT-B/16")`, vốn tải 335 MB mỗi phiên. `clip._download` kiểm
tra `~/.cache/clip/ViT-B-16.pt` bằng SHA256 trước, nên copy file vào đó là bỏ qua được
bước tải. Không có cũng không sao, chỉ chậm hơn.

In [ ]:
CLIP_SHA256 = '5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f'
cache_dir = Path.home() / '.cache' / 'clip'
cache_dir.mkdir(parents=True, exist_ok=True)
target = cache_dir / 'ViT-B-16.pt'

# walk_dirs chu khong phai glob('**'): xem ghi chu symlink o muc 1.
source = next((d / 'ViT-B-16.pt' for d in walk_dirs(INPUT_ROOT)
               if (d / 'ViT-B-16.pt').exists()), None)
if target.exists():
    print('Đã có sẵn trong cache:', target)
elif source is None:
    print('Không thấy ViT-B-16.pt trong /kaggle/input.')
    print('CLIP sẽ tự tải khi chạy (cần Internet: On).')
else:
    print('Copy', source, '->', target)
    shutil.copy2(source, target)

if target.exists():
    import hashlib
    digest = hashlib.sha256(target.read_bytes()).hexdigest()
    print('SHA256 khớp:', digest == CLIP_SHA256)
    if digest != CLIP_SHA256:
        print('  KHÔNG khớp -> clip.load sẽ tải lại. Kiểm tra lại file trong dataset.')

## 4. Preflight

Kiểm tra mọi thứ trước khi tiêu hàng giờ GPU. Bốn nhóm, và ba nhóm sau quan trọng hơn
nhóm đầu:

1. Có đủ file không.
2. **Bản code có đủ cờ mới không** — `ucf_option_augment.py` phải có `--augment-task-loss`
   và `--lambda-auto-basis`. Thiếu là bạn đang chạy bản cũ, và khối A sẽ im lặng biến
   thành đúng khối đối chứng.
3. `utils/layers.py` đã vá chưa (bản cũ ghi cứng `.to("cuda")`).
4. Ground truth có chưa; thiếu thì mục 5.1 sinh lại.

In [ ]:
import csv
import importlib
from collections import Counter

import numpy as np
import torch

_preflight_done = False


def preflight(force=False):
    global _preflight_done
    if _preflight_done and not force:
        return True

    problems = []

    if not torch.cuda.is_available():
        problems.append('Không có GPU. Settings -> Accelerator -> GPU T4 x2 hoặc P100.')

    if FEATURE_ROOT is None:
        print_input_tree()
        problems.append('Không dò ra dataset feature. Đặt FEATURE_OVERRIDE ở mục 1.')

    need = [SRC_DIR / n for n in
            ['model.py', 'ucf_train_augment.py', 'ucf_option_augment.py',
             'ucf_shift_sensitivity.py', 'ucf_test_description.py',
             'utils/dataset_augment.py', 'utils/tools.py', 'utils/layers.py',
             'utils/ucf_detectionMAP.py', 'clip/clip.py',
             'clip/bpe_simple_vocab_16e6.txt.gz',
             'tests/test_dataset_augment.py', 'tests/test_shift_consistency_loss.py',
             'tests/test_two_view_batching.py', 'tests/test_train_smoke.py']]
    need += [Path(TRAIN_LIST), Path(TEST_LIST), LIST_DIR / 'make_gt_ucf_relative.py',
             LIST_DIR / 'Temporal_Anomaly_Annotation.txt']
    for p in need:
        if not p.exists():
            problems.append(f'Thiếu file: {p}')

    # --- Bản code có đủ cờ không ------------------------------------------------------
    # 'augment_task_loss' và 'lambda_auto_basis' là hai cờ notebook này dựa vào. Thiếu
    # chúng thì khối A lặng lẽ biến thành đúng khối đối chứng, và khối B mất một nửa.
    if (SRC_DIR / 'ucf_option_augment.py').exists():
        import ucf_option_augment
        importlib.reload(ucf_option_augment)
        known = {action.dest for action in ucf_option_augment.parser._actions}
        needed = {'augment_task_loss', 'lambda_auto_basis',
                  'shift_ratio', 'shift_direction', 'shift_ratio_warmup', 'lambda_auto',
                  'lambda_auto_steps', 'lambda_auto_recalibrate', 'lambda_auto_max_growth',
                  'select_metric', 'run_tag', 'metrics_csv',
                  'deterministic', 'skip_shifted_view'}
        for name in sorted(needed - known):
            how = ('Code lấy từ GitHub: bạn CHƯA push bản mới lên branch '
                   f'{GITHUB_BRANCH}. Commit + push từ máy, xoá /kaggle/working/vadclip '
                   'và /kaggle/working/repo, rồi chạy lại mục 1.'
                   if CODE_FROM_GITHUB else
                   'Cập nhật Dataset code (New Version), xoá /kaggle/working/vadclip, '
                   'chạy lại mục 1.')
            problems.append(f'ucf_option_augment.py thiếu --{name.replace("_", "-")} — bản cũ. ' + how)

    # --- test() có trả về đủ chỉ số không ----------------------------------------------
    # LAST_METRICS là thứ đưa AUC nhánh A và mAP vào CSV. Không có nó thì hai cột đó rỗng.
    try:
        import ucf_test_description
        importlib.reload(ucf_test_description)
        if not hasattr(ucf_test_description, 'LAST_METRICS'):
            problems.append('ucf_test_description.py thiếu LAST_METRICS — bản cũ. '
                            'CSV sẽ không có auc_branch_a và avg_map.')
    except Exception as error:
        problems.append(f'Không nạp được ucf_test_description.py: {error!r}')

    # --- utils/layers.py đã vá chưa ---------------------------------------------------
    try:
        from utils.layers import DistanceAdj
        probe = DistanceAdj()                 # tham số nằm trên CPU
        if probe(2, 32).device.type != 'cpu':
            problems.append('utils/layers.py là BẢN CŨ: DistanceAdj ghi cứng .to("cuda").')
        del probe
    except Exception as error:
        problems.append(f'Không nạp được utils/layers.py: {error!r}')

    # --- Ground truth -----------------------------------------------------------------
    global GT_MISSING
    GT_MISSING = [str(LIST_DIR / n) for n in
                  ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')
                  if not (LIST_DIR / n).exists()]

    # --- Feature ----------------------------------------------------------------------
    if FEATURE_ROOT is not None:
        for name, expected in [('ucf_CLIP_rgb_relative.csv', 16100),
                               ('ucf_CLIP_rgbtest_relative.csv', 290)]:
            path = LIST_DIR / name
            if not path.exists():
                continue
            rows = list(csv.DictReader(open(path, encoding='utf-8')))
            missing = [r for r in rows if not (FEATURE_ROOT / r['path']).exists()]
            print(f'  {name}: {len(rows)} dòng (mong đợi {expected}), thiếu {len(missing)}')
            if missing:
                print('    Thiếu theo nhãn:', dict(Counter(r['label'] for r in missing)))
                for r in missing[:5]:
                    print('      ', r['path'])
                problems.append(f'{name}: thiếu {len(missing)} file feature.')

    if problems:
        print()
        print('PREFLIGHT KHÔNG ĐẠT:')
        for p in problems:
            print('  -', p)
        raise RuntimeError('Sửa các mục trên rồi chạy lại cell này.')

    print()
    print('PREFLIGHT ĐẠT')
    print('  GPU          :', torch.cuda.get_device_name(0))
    print('  torch        :', torch.__version__)
    print('  FEATURE_ROOT :', FEATURE_ROOT)
    print('  Bản code     : có --augment-task-loss và --lambda-auto-basis')
    print('  test()       : có LAST_METRICS -> CSV sẽ đủ 5 chỉ số')
    print('  layers.py    : bản đã vá')
    free = shutil.disk_usage(WORK).free / 1e9
    print(f'  /kaggle/working còn trống: {free:.1f} GB')
    if GT_MISSING:
        print('  Thiếu ground truth (mục 5.1 sẽ sinh lại):')
        for path in GT_MISSING:
            print('     ', path)
    else:
        gt = np.load(LIST_DIR / 'gt_ucf.npy')
        print('  gt_ucf.npy   :', len(gt), 'frame |', int(gt.sum()), 'frame bất thường')

    _preflight_done = True
    return True


preflight(force=True)

## 5. Hàm Chạy Lệnh

`build_train_cmd` nhận đúng một `dict` mô tả lần chạy, lấy từ `RUN_REGISTRY` ở mục 1.
Không có tham số nào bị giấu: mọi cờ ảnh hưởng tới kết quả đều được truyền tường minh,
kể cả khi giá trị trùng mặc định của script. Bản cũ cố ý bỏ trống một số cờ để "dùng
mặc định" — điều đó khiến đọc log không biết lần chạy thực sự dùng gì.

Ba cờ luôn cố định trong notebook này:

- `--select-metric auc` — lưu checkpoint tốt nhất theo AUC nhánh C, đúng yêu cầu.
- `--eval-steps 1280` — cadence của `ucf_train.py` gốc, ~12 lần chấm mỗi epoch.
- `--metrics-csv` — một dòng cho mỗi lần chấm, cột đặt tên rõ ràng theo nhánh.

In [ ]:
import time


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def co_cat(ten_cut):
    """Bốn cờ mô tả kiểu cắt. Luôn truyền đủ bốn, kể cả khi trùng mặc định."""
    cut = CUTS[ten_cut]
    if cut.get('skip'):
        # Không có L_shift và không augment thì khung nhìn dịch không ảnh hưởng gradient,
        # chỉ tốn gấp đôi batch thị giác. Script từ chối cờ này nếu lambda khác 0.
        return ['--skip-shifted-view', 'true']
    return [
        '--shift-offset',    str(cut.get('offset', 26)),
        '--shift-ratio',     str(cut.get('ratio', 0.0)),
        '--shift-direction', cut.get('direction', 'head'),
        '--random-shift',    'true' if cut.get('random') else 'false',
    ]


def co_lambda(ten_lam):
    """Ba–năm cờ mô tả cách tính lambda."""
    lam = LAMBDAS[ten_lam]
    if lam['kind'] == 'off':
        return ['--lambda-consistency', '0', '--lambda-auto', '0']
    if lam['kind'] == 'fixed':
        return ['--lambda-consistency', str(lam['value']), '--lambda-auto', '0']
    return [
        '--lambda-consistency',       '0',
        '--lambda-auto',              str(lam['ratio']),
        '--lambda-auto-basis',        lam['basis'],
        '--lambda-auto-steps',        '100',
        '--lambda-auto-recalibrate',  'true' if lam.get('recal') else 'false',
        '--lambda-auto-max-growth',   '2.0',
    ]


def build_train_cmd(tag, metrics_csv=True, extra=None):
    """Dòng lệnh đầy đủ cho một tag trong RUN_REGISTRY. Không cờ nào bị giấu."""
    run = RUN_REGISTRY[tag]
    return PY + [
        'ucf_train_augment.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list',   TRAIN_LIST,
        '--test-list',    TEST_LIST,
        *GT_ARGS,
        '--seed',                 str(SEED),
        '--max-epoch',            str(MAX_EPOCH),
        '--lr',                   LR,
        '--batch-size',           str(BATCH_SIZE),
        '--use-pretrained-model', 'false' if not USE_PRETRAINED else 'true',
        '--pretrained-model-path', str(PAPER_MODEL or ''),
        # --- cái đang thử ---
        *co_cat(run['cut']),
        '--augment-task-loss',    'true' if run['augment'] else 'false',
        *co_lambda(run['lam']),
        '--consistency-branch',   BRANCH,
        '--consistency-detach',   'false',
        '--consistency-warmup',   str(WARMUP_EPOCHS),
        '--shift-ratio-warmup',   '0',
        # --- luật chấm & chọn ---
        '--eval-steps',           str(EVAL_STEPS),
        '--select-metric',        SELECT_METRIC,   # 'auc' = giữ checkpoint AUC cao nhất
        # --- nơi ghi ---
        '--output-model-path',    str(MODEL_DIR / f'model_{tag}.pth'),
        '--checkpoint-path',      str(SCRATCH / f'checkpoint_{tag}.pth'),
        '--save-cur-path',        str(SCRATCH / f'model_cur_{tag}.pth'),
        '--epoch-checkpoint-dir', str(SCRATCH / f'epoch_checkpoints_{tag}'),
        '--run-tag',              tag,
        '--metrics-csv',          METRICS_CSV if metrics_csv else '',
        '--num-workers',          str(NUM_WORKERS),
        '--pin-memory',           'true',
    ] + [str(x) for x in (extra or [])]


def train_run(tag):
    """Bỏ qua lần chạy đã có trọng số, kể cả trọng số của phiên Kaggle trước."""
    preflight()
    da_co = duong_dan_model(tag)
    if da_co is not None:
        print(f'[bỏ qua] {tag}: đã có trọng số tại {da_co}')
        return None
    run = RUN_REGISTRY[tag]
    print('#' * 94)
    print(f'{tag}  |  khối {run["khoi"]}  |  cắt: {CUTS[run["cut"]]["nhan"]}  |  '
          f'augment: {"có" if run["augment"] else "không"}  |  '
          f'lambda: {LAMBDAS[run["lam"]]["nhan"]}')
    print('#' * 94)
    started = time.time()
    output = run_command(build_train_cmd(tag), log_name=f'train_{tag}.log')
    print(f'Xong sau {(time.time() - started) / 3600:.2f} giờ')
    return output


def train_block(tags):
    """Chạy lần lượt, không để một lần chạy hỏng làm mất các lần còn lại."""
    loi = []
    for tag in tags:
        try:
            train_run(tag)
        except Exception as error:
            loi.append((tag, repr(error)))
            print(f'[LỖI] {tag}: {error}')
    if loi:
        print()
        print('Thất bại:')
        for tag, error in loi:
            print(f'  {tag}: {error}')
    return loi


def shift_sensitivity(tag, model_path=None, offsets=(0, 8, 16, 32)):
    """Tương quan điểm số trước và sau khi dịch, sau khi căn chỉnh lại.

    KHÔNG cùng loại với auc_branch_c: chạy trên 266 video và chỉ vùng chồng lấn.
    """
    preflight()
    output_dir = RESULT_DIR / f'shift_sens_{tag}'
    if (output_dir / 'shift_sensitivity_summary.csv').exists():
        print(f'[đã có] {output_dir}')
        return output_dir
    duong_dan = model_path or duong_dan_model(tag)
    if duong_dan is None:
        print(f'[chưa có trọng số] {tag}')
        return None
    run_command(PY + [
        'ucf_shift_sensitivity.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list',    TEST_LIST,
        '--model-path',   str(duong_dan),
        '--gt-path',      GT_ARGS[1],
        '--offsets',      *[str(o) for o in offsets],
        '--output-dir',   str(output_dir),
    ], log_name=f'sens_{tag}.log')
    return output_dir


def in_thu_lenh(tag):
    """In dòng lệnh đầy đủ, ghép cờ với giá trị của nó trên cùng một dòng."""
    cmd = [str(p) for p in build_train_cmd(tag)]
    print(f'--- {tag} ---')
    i = 0
    while i < len(cmd):
        if cmd[i].startswith('--') and i + 1 < len(cmd) and not cmd[i + 1].startswith('--'):
            print(f'  {cmd[i]:<26} {cmd[i + 1]}')
            i += 2
        else:
            print(f'  {cmd[i]}')
            i += 1


print('Sẵn sàng.')
print()
print('Dòng lệnh đầy đủ của lần chạy đầu tiên (chỉ in, không chạy):')
in_thu_lenh(RUNS_THIS_SESSION[0])

### 5.1. Sinh Lại Ground Truth (chỉ khi mục 4 báo thiếu)

Ba file `gt_*.npy` sinh từ file nhãn thời gian cộng với độ dài thật của từng file đặc
trưng, nên bắt buộc phải có feature trước.

In [ ]:
if GT_MISSING:
    run_command(PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        # Bắt buộc: mặc định của script là 'list/Temporal_Anomaly_Annotation.txt'
        # tính theo thư mục hiện hành, mà thư mục hiện hành là src/ -> không có.
        '--annotation', str(LIST_DIR / 'Temporal_Anomaly_Annotation.txt'),
        '--output-dir', LIST_DIR,
    ], log_name='make_gt.log')
    preflight(force=True)
else:
    print('Đã có đủ ground truth, bỏ qua.')

## 6. Unit Test

Sáu nhóm kiểm tra, khoảng một–hai phút, chạy trên CPU với bộ mã hoá CLIP giả nên không
cần feature thật.

Hai nhóm mới đáng chú ý, vì chúng kiểm đúng hai cờ mà notebook này dựa vào:

- `--augment-task-loss` có thật sự chạm vào `L₁`/`L₂` không (và **không** chạm vào `L₃`,
  vốn chỉ phụ thuộc prompt văn bản).
- `--lambda-auto-basis grad` có giải ra một λ **khác** với `loss` không. Nếu hai giá trị
  bằng nhau thì nhánh mới đã lặng lẽ rơi về nhánh cũ.

Test đỏ ở đây là dừng lại, đừng chạy tiếp mục 8.

In [ ]:
for test_file in ('test_dataset_augment', 'test_shift_consistency_loss',
                  'test_two_view_batching', 'test_train_smoke'):
    run_command(PY + [f'tests/{test_file}.py'], log_name=f'{test_file}.log')

## 7. Ước Lượng Thời Gian — CHẠY TRƯỚC MỤC 8

Hai mươi bước huấn luyện thật, cộng một phép đo chi phí chấm điểm. Rẻ (vài phút) và trả
lời đúng câu hỏi mà không nên đoán: **mấy lần chạy thì kịp 12 giờ?**

Chi phí không đồng đều giữa các lần chạy:

- Lần chạy **có** khung nhìn dịch đưa `2B × 256 × 512` qua model → đắt hơn.
- Lần chạy **Ctrl** dùng `--skip-shifted-view` nên chỉ có `B × 256 × 512` → rẻ hơn rõ rệt.
- `--lambda-auto-basis grad` thêm hai lượt `autograd.grad` ở **vài** bước hiệu chỉnh,
  không phải mọi bước, nên chi phí không đáng kể.

Cell này ước lượng riêng hai mức rồi cộng theo đúng danh mục bạn chọn.

In [ ]:
PROBE_STEPS = 20
STEPS_PER_EPOCH = 125          # min(8000, 8100) // 64, với drop_last


def do_toc_do(tag, steps):
    started = time.time()
    run_command(build_train_cmd(tag, metrics_csv=False,
                                extra=['--debug-max-steps', str(steps)]),
                log_name=f'probe_{tag}.log')
    return (time.time() - started) / steps


# Hai mức chi phí: có khung nhìn dịch (2 lượt qua model) và không có (1 lượt).
tag_2view = next((t for t in RUN_REGISTRY if not CUTS[RUN_REGISTRY[t]['cut']].get('skip')), None)
tag_1view = next((t for t in RUN_REGISTRY if CUTS[RUN_REGISTRY[t]['cut']].get('skip')), None)

print('=' * 94)
print(f'--- {PROBE_STEPS} bước, CÓ khung nhìn dịch ({tag_2view}) ---')
per_step_2view = do_toc_do(tag_2view, PROBE_STEPS)

per_step_1view = per_step_2view
if tag_1view:
    print()
    print('=' * 94)
    print(f'--- {PROBE_STEPS} bước, KHÔNG khung nhìn dịch ({tag_1view}) ---')
    per_step_1view = do_toc_do(tag_1view, PROBE_STEPS)

# Chi phí một lần chấm: chạy 11 bước với eval bật. Bước thứ 10 kích hoạt đúng một lần
# chấm (step = i * BATCH_SIZE * 2, i = 10 -> 1280 = EVAL_STEPS), nên hiệu số so với chi
# phí thuần huấn luyện chính là chi phí một lần chấm.
print()
print('=' * 94)
print('--- chi phí một lần chấm trên 290 video test ---')
started = time.time()
run_command(build_train_cmd(tag_2view, metrics_csv=False,
                            extra=['--debug-max-steps', '11']),
            log_name='probe_eval.log')
eval_seconds = max(0.0, (time.time() - started) - per_step_2view * 11)

# Lần chạy thật: ~12 lần chấm giữa epoch (--eval-steps) + 1 lần cuối epoch (--metrics-csv).
evals_per_epoch = (STEPS_PER_EPOCH * BATCH_SIZE * 2) // EVAL_STEPS + 1
evals_total = evals_per_epoch * MAX_EPOCH
eval_h = eval_seconds * evals_total / 3600


def gio_uoc_luong(tag):
    per_step = per_step_1view if CUTS[RUN_REGISTRY[tag]['cut']].get('skip') else per_step_2view
    return per_step * STEPS_PER_EPOCH * MAX_EPOCH / 3600 + eval_h


print()
print('=' * 94)
print(f'{STEPS_PER_EPOCH} bước/epoch · {evals_per_epoch} lần chấm/epoch · '
      f'{evals_total} lần chấm mỗi lần chạy')
print(f'{per_step_2view:.2f} s/bước (2 view) · {per_step_1view:.2f} s/bước (1 view) · '
      f'{eval_seconds:.0f} s mỗi lần chấm')
print()
print(f'ƯỚC LƯỢNG CHO {MAX_EPOCH} EPOCH — ngân sách phiên: {TIME_BUDGET_HOURS} giờ')
print('-' * 94)
tong = 0.0
vua = []
for tag in RUNS_THIS_SESSION:
    if duong_dan_model(tag) is not None:
        print(f'  {tag:<20} đã xong ở phiên trước')
        continue
    gio = gio_uoc_luong(tag)
    tong += gio
    trang_thai = 'vừa' if tong <= TIME_BUDGET_HOURS else 'TRÀN'
    if tong <= TIME_BUDGET_HOURS:
        vua.append(tag)
    print(f'  {tag:<20} {gio:5.2f} giờ   cộng dồn {tong:5.2f} giờ   {trang_thai}')
print('-' * 94)
if tong > TIME_BUDGET_HOURS:
    print(f'⚠ Danh mục phiên này cần {tong:.1f} giờ, quá ngân sách {TIME_BUDGET_HOURS} giờ.')
    print(f'  Vừa trong ngân sách: {vua}')
    print('  Sửa RUNS_THIS_SESSION ở mục 1 rồi chạy lại từ mục 1.')
    print('  (Script không resume được giữa chừng — bị cắt là mất cả lần chạy đó.)')
else:
    print(f'Tổng {tong:.1f} giờ, nằm trong ngân sách. 20 bước đầu vẫn lạc quan hơn thực tế,')
    print('nên đừng để sát mép.')

## 8. Khối A — Cắt Đặc Trưng Như Dữ Liệu Tăng Cường (không có `L_shift`)

### Khối này hỏi gì

Nếu chỉ đưa thêm bản dịch thời gian vào như một mẫu huấn luyện nữa — không ràng buộc hai
khung nhìn phải giống nhau — thì model có tốt lên không? Và **kiểu cắt nào** đáng dùng?

Đây là câu hỏi phải trả lời **trước** khối B. Nếu augmentation một mình đã đủ, thì phần
`L_shift` phải chứng minh nó thêm được gì ngoài phần đó. Bản cũ không tách được hai
chuyện này.

### Sáu cấu hình

| Tag | Cắt | Ý nghĩa |
|---|---|---|
| `ctrl_nocut` | không cắt | Mốc so. Không augment, không `L_shift`. |
| `aug_head26` | bỏ 26 ô đầu | Đúng kiểu cắt của vòng 1. Mất hẳn 26 ô nội dung đầu video. |
| `aug_tail26` | đệm 26 ô vào đầu | Đẩy nội dung lùi lại. **Không mất gì** với 72% video có `ℓ ≤ 256`. |
| `aug_both26r` | ±(1…26) ngẫu nhiên | Không thiên vị đầu hay cuối, và ràng buộc ở nhiều khoảng cách chứ không chỉ một. |
| `aug_ratio10` | ±10% độ dài **của chính video đó** | 26 ô là ~10% của *lưới*, nhưng với video trung vị (ℓ=138) nó là ~19%. Cắt theo tỉ lệ làm đều liều lượng. |
| `aug_ratio20r` | ±(1…20%) ngẫu nhiên | Liều cao hơn, vẫn theo tỉ lệ. |

### Một cảnh báo thật về nhãn

Với `head`, 26 ô đầu bị **vứt đi**. Video bất thường mà sự kiện nằm trọn trong 26 ô đầu
sẽ thành một mẫu toàn cảnh bình thường nhưng vẫn mang nhãn bất thường — tức nhiễu nhãn.
`tail` không có vấn đề này (không mất nội dung khi `ℓ + Δ ≤ 256`), nên nếu `aug_tail26`
thắng `aug_head26` thì đây là lời giải thích đầu tiên nên nghĩ tới.

Script đã bỏ khỏi phần loss những item mà khung nhìn dịch không còn nội dung hợp lệ
(`ℓ − Δ ≤ 0`), nhưng nó **không** biết sự kiện nằm ở đâu bên trong video — đó là thứ dữ
liệu giám sát yếu không cho biết.

In [ ]:
tags_A = [t for t in RUNS_THIS_SESSION
          if RUN_REGISTRY[t]['khoi'] in ('Ctrl', 'A')]
print('Khối Ctrl + A phiên này:', tags_A)
print()
train_block(tags_A)

## 9. Khối B — Tính Trọng Số λ Thế Nào Cho Không Phải Đoán

### Vấn đề

`L = L₁ + L₂ + L₃ + λ·L_shift`. Ba số hạng đầu có hệ số 1, số hạng thứ tư thì không ai
biết nên để bao nhiêu. Vòng 1 đặt `λ = 0,01` bằng tay. Đó là con số đoán, và tệ hơn: nó
**không mang ý nghĩa gì** vì `L_shift` ở bước 0 chỉ khoảng `3,5 × 10⁻³`, tức nhỏ hơn
`L_task ≈ 2,11` khoảng 600 lần. Nhân thêm 0,01 nữa thì số hạng thứ tư chiếm khoảng
**một phần sáu mươi nghìn** tổng hàm mục tiêu.

### Năm cách làm thay cho đoán

**1. Đặt tay một con số tuyệt đối** — `--lambda-consistency 0.01`.
Đây là cái đang bị thay thế. Vấn đề không phải là nó sai, mà là nó **không chuyển được**:
đổi kiểu cắt thì `L_shift` đổi thang đo, và cùng một λ bỗng có nghĩa hoàn toàn khác.

**2. Đặt theo *tỉ lệ giá trị loss*** — `--lambda-auto 0.01 --lambda-auto-basis loss`.
Chạy vài chục bước với số hạng tắt, đo cả hai vế, rồi giải
`λ = r · L_task / L_shift`. Giờ `r` mới là thứ bạn chọn, và `r = 0,01` nghĩa là "cho số
hạng này chiếm 1% hàm mục tiêu" — một câu có nghĩa ở mọi cấu hình cắt.
*Nhược điểm đã đo được ở vòng 2:* giải một lần rồi giữ nguyên thì tỉ lệ trôi. Ba lần chạy
đặt mục tiêu 1% / 3% / 10% kết thúc ở **1,9% / 2,6% / 4,0%** — định trải một bậc thập
phân, cuối cùng chỉ còn cách nhau hai lần. Không tách được gì.

**3. Giải lại mỗi epoch** — thêm `--lambda-auto-recalibrate true`.
Sửa đúng cái trôi ở trên. Có `--lambda-auto-max-growth` làm dây cương, vì giữ một tỉ lệ
cố định khi mẫu số đang tiến về 0 sẽ đòi λ vô hạn.

**4. Đặt theo *chuẩn gradient*** — `--lambda-auto-basis grad`. **Cách đúng về nguyên tắc.**
Optimizer không "cảm" được giá trị loss, nó chỉ đi theo gradient. Hai số hạng có giá trị
chênh nhau 600 lần vẫn có thể cho gradient tương đương, hoặc ngược lại. Cờ này giải
`λ = r · ‖∂L_task/∂θ‖ / ‖∂L_shift/∂θ‖`, đo bằng hai lượt `autograd.grad` tại bước hiệu
chỉnh. Đây chính là ý tưởng của **GradNorm**, rút gọn còn một hệ số vô hướng.

**5. Tìm λ trên một tập validation cắt từ train** — *chưa làm, và đây là lỗ hổng còn lại.*
Bốn cách trên đều chỉ trả lời "λ bao nhiêu thì hai số hạng cân nhau", chứ không trả lời
"λ bao nhiêu thì model tốt nhất". Muốn trả lời câu sau một cách sạch thì phải chọn trên
một tập validation **cắt theo video** từ tập train, không phải trên test. Notebook này
chưa làm được điều đó; khi báo cáo phải ghi thẳng là siêu tham số được chọn trên test.

> **Còn *uncertainty weighting* (Kendall & Gal) thì sao?** Học `λ = 1/(2σ²)` cùng với
> trọng số model. Nó hợp với **nhiều tác vụ** có likelihood riêng, nhưng `L_shift` là một
> **số hạng chính quy hoá** có thể bị ép về 0 — mà ép nó về 0 chính là mục tiêu. Khi mẫu
> số tiến về 0, bài toán tối ưu σ không còn điểm dừng có nghĩa. Vì vậy tôi không đưa nó
> vào; đây là lựa chọn có chủ ý, không phải bỏ sót.

### Sáu cấu hình

| Tag | Cách tính λ | Trả lời |
|---|---|---|
| `lam_fixed001` | tay, tuyệt đối 0,01 | Tái lập `v0` vòng 1, làm mốc |
| `lam_loss001` | tỉ lệ loss 1%, giải một lần | Tái lập `v2_lam0.01` vòng 2 |
| `lam_loss001_recal` | tỉ lệ loss 1%, giải lại mỗi epoch | Sửa cái trôi có giúp không |
| `lam_grad001` | tỉ lệ **gradient** 1% | Đổi cơ sở đo có đổi kết luận không |
| `lam_grad003` | tỉ lệ gradient 3% | Đường cong liều |
| `lam_grad010` | tỉ lệ gradient 10% | Đường cong liều |

Cả sáu dùng **cùng một kiểu cắt** (`BASE_CUT` ở mục 1, mặc định `head26` để so được với
hai vòng trước) và **tắt** `--augment-task-loss`, để thứ duy nhất thay đổi là λ.

In [ ]:
tags_B = [t for t in RUNS_THIS_SESSION if RUN_REGISTRY[t]['khoi'] == 'B']
print('Khối B phiên này:', tags_B)
print()
train_block(tags_B)

## 10. Đo Độ Nhạy Dịch Chuyển

Phép đo trả lời đúng câu hỏi mà `L_shift` nhắm tới: dịch đầu vào đi Δ bước, căn kết quả
trở lại trục cũ, thì chuỗi điểm số có giữ nguyên không? Không huấn luyện lại gì cả, chỉ
chấm điểm, nên rẻ.

Ở vòng 1 đây là **bằng chứng có ý nghĩa duy nhất**: AUC và mAP thay đổi trong phạm vi
dao động giữa các lần chạy, còn tương quan thì cải thiện nhất quán ở cả ba mức dịch.

In [ ]:
# Mốc tham chiếu: checkpoint của tác giả.
if PAPER_MODEL and Path(PAPER_MODEL).exists():
    try:
        shift_sensitivity('paper', model_path=PAPER_MODEL)
    except Exception as error:
        print('[LỖI] paper:', error)
else:
    print('Không có model_ucf.pth — bỏ qua mốc đối chiếu.')

# Mọi lần chạy đã có trọng số, kể cả của phiên trước.
for tag in RUN_REGISTRY:
    if duong_dan_model(tag) is None:
        continue
    print('=' * 94)
    try:
        shift_sensitivity(tag)
    except Exception as error:
        print(f'[LỖI] {tag}: {error}')

## 11. Bảng Kết Quả

Ba bảng **tách rời**, đúng như quy ước ở đầu notebook. Không bảng nào trộn chỉ số của
bảng khác vào cùng một cột.

- **Bảng 1 — chất lượng phát hiện.** `sel_*` là số của file trọng số đã lưu (lần chấm có
  AUC nhánh C cao nhất); `end_*` là cuối epoch cuối.
- **Bảng 2 — ổn định dưới dịch chuyển.** Chạy trên 266 video, chỉ vùng chồng lấn.
  Không so với Bảng 1.
- **Bảng 3 — sàn nhiễu.** Độ lệch chuẩn giữa các lần chạy đối chứng. Mọi hiệu ứng nhỏ hơn
  con số này đều **chưa** kết luận được.

Sàn nhiễu đã biết từ hai vòng trước: **0,58 điểm AUC** và **1,55 điểm mAP** giữa hai lần
chạy giống hệt nhau về mặt toán học. Giữ con số đó trong đầu khi đọc Bảng 1.

In [ ]:
import pandas as pd

OFFSETS = [8, 16, 32]

# CSV cũ dùng tên cột mơ hồ 'auc'/'ap'. Đổi tên khi nạp để mọi nguồn cùng một lược đồ.
DOI_TEN_CU = {'auc': 'auc_branch_c', 'ap': 'ap_branch_c'}

SO_COT = ['epoch', 'step', 'auc_branch_c', 'ap_branch_c',
          'auc_branch_a', 'ap_branch_a', 'avg_map', 'lambda_used']


def nap_metrics(*duong_dan):
    """Gộp mọi CSV thành một bảng, một dòng cho mỗi lần chấm."""
    khung = []
    for path in duong_dan:
        if not path or not Path(path).exists():
            continue
        frame = pd.read_csv(path)
        frame = frame[frame['run'] != 'run']              # dòng tiêu đề bị lặp khi nối file
        frame = frame.rename(columns={k: v for k, v in DOI_TEN_CU.items()
                                      if k in frame.columns and v not in frame.columns})
        frame['nguon'] = Path(path).name
        khung.append(frame)
    if not khung:
        return pd.DataFrame()
    frame = pd.concat(khung, ignore_index=True)
    for cot in SO_COT:
        if cot in frame.columns:
            frame[cot] = pd.to_numeric(frame[cot], errors='coerce')
    if 'eval_kind' not in frame.columns:
        frame['eval_kind'] = 'khong_ro'                   # CSV cũ không có cột này
    # Khử trùng theo (run, epoch, step): chạy lại thì CSV được NỐI THÊM, không ghi đè.
    return frame.drop_duplicates(subset=['run', 'epoch', 'step'], keep='last')


def bang_chat_luong(frame):
    """Bảng 1: một dòng mỗi lần chạy. sel_* là bộ trọng số đã lưu, end_* là cuối cùng."""
    dong = []
    for tag, nhom in frame.groupby('run'):
        nhom = nhom.sort_values(['epoch', 'step'])
        # sel_*: lần chấm có AUC nhánh C cao nhất. Đúng thứ --select-metric auc đã lưu.
        # Chỉ xét các lần chấm GIỮA epoch, vì chỉ chúng mới cập nhật checkpoint; lần chấm
        # cuối epoch chỉ ghi log (xem ucf_train_augment.py).
        ung_vien = nhom[nhom['eval_kind'] != 'epoch_end']
        if ung_vien.empty:
            ung_vien = nhom
        tot = ung_vien.loc[ung_vien['auc_branch_c'].idxmax()]
        cuoi = nhom.iloc[-1]
        run = RUN_REGISTRY.get(tag)
        dong.append({
            'run': tag,
            'khoi': run['khoi'] if run else '?',
            'cat': CUTS[run['cut']]['nhan'] if run else '?',
            'augment': ('có' if run['augment'] else 'không') if run else '?',
            'lambda': LAMBDAS[run['lam']]['nhan'] if run else '?',
            'lambda_used': float(cuoi.get('lambda_used', float('nan'))),
            'sel_auc_c': float(tot['auc_branch_c']) * 100,
            'sel_ap_c': float(tot['ap_branch_c']) * 100,
            'sel_auc_a': float(tot.get('auc_branch_a', float('nan'))) * 100,
            'sel_map': float(tot.get('avg_map', float('nan'))),
            'sel_epoch': int(tot['epoch']),
            'sel_step': int(tot['step']),
            'end_auc_c': float(cuoi['auc_branch_c']) * 100,
            'end_map': float(cuoi.get('avg_map', float('nan'))),
            'so_lan_cham': len(nhom),
        })
    return pd.DataFrame(dong).set_index('run').sort_values(['khoi', 'run'])


def nap_do_nhay(tag):
    """Bảng 2: chạy trên 266 video, chỉ vùng chồng lấn. KHÔNG so với Bảng 1."""
    ung_vien = [RESULT_DIR / f'shift_sens_{tag}'] + [d for d in PRIOR_RESULT_DIRS
                                                     if d.name == f'shift_sens_{tag}']
    for thu_muc in ung_vien:
        path = thu_muc / 'shift_sensitivity_summary.csv'
        if not path.exists():
            continue
        frame = pd.read_csv(path).set_index('offset')
        dong = {f'corr_d{o}': float(frame.loc[o, 'mean_classifier_corr'])
                for o in OFFSETS if o in frame.index}
        dong['auc_spread_shift'] = float(frame.iloc[0]['classifier_auc_spread_common'])
        return dong
    return None


metrics = nap_metrics(METRICS_CSV, *PRIOR_METRICS)

print('=' * 100)
print('BẢNG 1 — CHẤT LƯỢNG PHÁT HIỆN (290 video test, toàn bộ khung)')
print('=' * 100)
if metrics.empty:
    print('Chưa có CSV nào. Chạy mục 9/10 trước.')
    bang1 = pd.DataFrame()
else:
    bang1 = bang_chat_luong(metrics)
    print(bang1.round(2).to_string())
    bang1.to_csv(RESULT_DIR / 'bang1_chat_luong.csv')
    print()
    print('sel_* = lần chấm có auc_branch_c cao nhất = ĐÚNG bộ trọng số đã lưu ra file.')
    print('end_* = lần chấm cuối cùng. Chênh lệch sel − end là phần do luật chọn tạo ra.')
    print()
    print('Chênh lệch sel − end:')
    for tag in bang1.index:
        r = bang1.loc[tag]
        print(f"  {tag:<20} sel {r['sel_auc_c']:6.2f} (epoch {int(r['sel_epoch'])}, "
              f"step {int(r['sel_step'])})  |  end {r['end_auc_c']:6.2f}  |  "
              f"chênh {r['sel_auc_c'] - r['end_auc_c']:+5.2f}  |  "
              f"{int(r['so_lan_cham'])} lần chấm")

print()
print('=' * 100)
print('BẢNG 2 — ỔN ĐỊNH DƯỚI DỊCH CHUYỂN (266 video, CHỈ vùng chồng lấn)')
print('=' * 100)
print('Không cùng thang đo với Bảng 1. corr càng gần 1 càng ổn định.')
print()
dong_ds = []
for tag in ['paper'] + list(RUN_REGISTRY):
    ds = nap_do_nhay(tag)
    if ds:
        dong_ds.append({'run': tag, **ds})
if dong_ds:
    bang2 = pd.DataFrame(dong_ds).set_index('run')
    print(bang2.round(4).to_string())
    bang2.to_csv(RESULT_DIR / 'bang2_do_nhay.csv')
else:
    print('Chưa có bảng độ nhạy nào. Chạy mục 11 trước.')

print()
print('=' * 100)
print('BẢNG 3 — SÀN NHIỄU')
print('=' * 100)
print('Đã biết từ hai vòng trước: hai lần chạy giống hệt nhau về mặt toán học cho')
print('88,13 / 8,50 và 87,55 / 6,95 → sàn nhiễu 0,58 điểm AUC và 1,55 điểm mAP.')
print()
if not bang1.empty:
    doi_chung = [t for t in bang1.index if 'ctrl' in t]
    if len(doi_chung) >= 2:
        print(f'Phiên này có {len(doi_chung)} lần đối chứng:')
        for cot in ('sel_auc_c', 'sel_map'):
            gia_tri = bang1.loc[doi_chung, cot].dropna()
            if len(gia_tri) >= 2:
                print(f'  {cot:<12} trung bình {gia_tri.mean():7.3f}  '
                      f'độ lệch chuẩn {gia_tri.std():6.3f}  '
                      f'khoảng [{gia_tri.min():.2f}, {gia_tri.max():.2f}]')
    else:
        print('Chỉ có ≤1 lần đối chứng ở đây, nên chưa đo lại được sàn nhiễu.')
        print('Muốn đo: chạy ctrl_nocut thêm với seed 1234 và 2024.')
    print()
    print('QUY TẮC ĐỌC BẢNG 1: chênh lệch sel_auc_c dưới 0,58 điểm hoặc sel_map dưới')
    print('1,55 điểm thì KHÔNG được tuyên bố là cải thiện. Ghi thẳng điều đó vào báo cáo.')

## 12. Gom Sản Phẩm

Mọi thứ trong `/kaggle/working` tự thành Output của notebook. Cell này liệt kê và cảnh
báo nếu vượt hạn mức 20 GB.

**Để chạy tiếp ở phiên sau:** Save Version → phiên mới → Add Input → chọn chính notebook
này → mục 1 sẽ tự tìm thấy `shift_kaggle_metrics.csv` và các file `model_*.pth` cũ, và
mục 8/9 sẽ bỏ qua những lần chạy đã xong.

In [ ]:
total = 0
print('Nội dung /kaggle/working:')
for f in sorted(WORK.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        total += size
        if size > 1e6:
            print(f'  {size / 1e6:8.0f} MB  {f.relative_to(WORK)}')
print()
print(f'Tổng: {total / 1e9:.2f} GB / 20 GB')
if total > 18e9:
    print('  GẦN CHẠM HẠN MỨC. Xoá bớt file trong /kaggle/working/models.')

print()
print('File nhỏ (log, csv):')
for f in sorted(RESULT_DIR.rglob('*')):
    if f.is_file() and f.stat().st_size <= 1e6:
        print(f'  {f.stat().st_size / 1e3:8.0f} KB  {f.relative_to(WORK)}')

print()
print('Còn lại trong danh mục, chưa chạy:')
con_lai = [t for t in RUN_REGISTRY if duong_dan_model(t) is None]
print(' ', con_lai if con_lai else '(không còn — đã chạy hết 12 lần chạy)')
if con_lai:
    print()
    print('Phiên sau: Save Version → notebook mới → Add Input chính notebook này →')
    print('  RUNS_THIS_SESSION =', con_lai[:4])

## Ghi Chú

**Cờ mới mà notebook này dựa vào.** Hai cờ được thêm vào `ucf_train_augment.py`:

- `--augment-task-loss` — tính `L₁`/`L₂` trên cả khung nhìn dịch, lấy trung bình với
  khung nhìn đầy đủ. Item nào có `ℓ_shift = 0` thì bị loại khỏi nửa dịch, để không huấn
  luyện trên một tensor toàn 0 vẫn mang nhãn bất thường.
- `--lambda-auto-basis {loss,grad}` — `loss` giữ nguyên hành vi vòng 2; `grad` cân theo
  chuẩn gradient.

Cả hai đều **mặc định tắt / giữ hành vi cũ**, nên mọi dòng lệnh của vòng 1 và vòng 2 vẫn
tái lập y nguyên.

**Cột CSV đã đổi tên.** `auc` → `auc_branch_c`, và thêm `ap_branch_c`, `auc_branch_a`,
`ap_branch_a`, `avg_map`, `eval_kind`, cùng các cột ghi lại cấu hình. CSV cũ
(`shift_v2_metrics.csv`) vẫn đọc được: mục 11 tự đổi tên cột khi nạp.

**Vì sao `ctrl_nocut` dùng `--skip-shifted-view`.** Không có `L_shift` và không có
augmentation thì khung nhìn dịch không ảnh hưởng gì tới gradient, chỉ tốn gấp đôi batch
thị giác. `tests/test_two_view_batching.py` khẳng định CLIPVAD độc lập theo batch, nên
logits của khung nhìn đầy đủ giống hệt nhau ở cả hai đường.

**Chuyện notebook này KHÔNG giải quyết.** Siêu tham số vẫn được chọn trên tập test, vì
`--select-metric auc` chọn checkpoint theo AUC test và vì bạn sẽ đọc Bảng 1 để chọn kiểu
cắt. Cách sạch là cắt validation theo video từ tập train. Khi viết báo cáo, ghi thẳng
điều này thay vì để người đọc tự phát hiện.